# Mercy Fantasy

## Set Up

In [5]:
import yahoo_fantasy_api as yfa
import nflreadpy as nfl
import pandas as pd
from yahoo_oauth import OAuth2
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Float, Boolean, Date, BigInteger, DateTime
from datetime import datetime
from getParlays import get_parlays
import time

oauth = OAuth2(None, None, from_file="oauth2.json")
if not oauth.token_is_valid():
    oauth.refresh_access_token()
gm = yfa.Game(oauth, "nfl")

[2026-10-09 09:54:26,363 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-10-09 09:54:26,372 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 163.97793197631836
[2026-10-09 09:54:26,373 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-10-09 09:54:26,378 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 163.9842610359192
[2026-10-09 09:54:26,379 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


In [30]:
idMap = {2023: '423.l.927065', 2024: '449.l.978032', 2025: '461.l.1390291'}

In [10]:
# Database info
user = "mercyfantasy"
password = "Matthew23"
host = "localhost"
port = "5432"
database = "mercyfantasy"

# create a database connection
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

## NFL Reference

In [3]:
# Load schedule data for all seasons
schedules = nfl.load_schedules(seasons=True)   # includes past + current seasons

# Filter for seasons 2023 onward
schedules_df = schedules.to_pandas()
schedules_df = schedules_df[schedules_df.season >= 2023].reset_index(drop=True)  # adjust column name as needed

# Convert gameday
schedules_df["gameday"] = pd.to_datetime(schedules_df["gameday"], errors="coerce")

# FIX: Convert gametime into full datetime
schedules_df["gametime"] = pd.to_datetime(
    schedules_df["gameday"].dt.date.astype(str) + " " + schedules_df["gametime"].astype(str),
    errors="coerce"
)

# Regularize team names for consistency
schedules_df[['away_team', 'home_team']] = (
    schedules_df[['away_team', 'home_team']]
        .replace({'LA': 'LAR'})
)

NameError: name 'nfl' is not defined

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE nfl_schedule RESTART IDENTITY CASCADE;"))

schedules_df.to_sql(
    "nfl_schedule",
    engine,
    if_exists="append",
    index=False,
    dtype={
    "game_id": String(50),
    "season": Integer(),
    "game_type": String(10),
    "week": Integer(),
    "gameday": DateTime(),
    "weekday": String(10),
    "gametime": DateTime(),

    "away_team": String(10),
    "away_score": Integer(),
    "home_team": String(10),
    "home_score": Integer(),
    "location": String(100),
    "result": Float(),
    "total": Float(),
    "overtime": Boolean(),

    # IDs — must be STRING
    "old_game_id": String(50),
    "gsis": String(50),
    "nfl_detail_id": String(50),
    "pfr": String(50),
    "pff": String(50),
    "espn": String(50),
    "ftn": String(50),

    "away_rest": Float(),
    "home_rest": Float(),
    "away_moneyline": Float(),
    "home_moneyline": Float(),
    "spread_line": Float(),
    "away_spread_odds": Float(),
    "home_spread_odds": Float(),
    "total_line": Float(),
    "under_odds": Float(),
    "over_odds": Float(),

    "div_game": Boolean(),
    "roof": String(50),
    "surface": String(50),
    "temp": Float(),
    "wind": Float(),

    # QB / Coach / Stadium — keep as STRING
    "away_qb_id": String(20),
    "home_qb_id": String(20),
    "away_qb_name": String(100),
    "home_qb_name": String(100),
    "away_coach": String(100),
    "home_coach": String(100),
    "referee": String(100),
    "stadium_id": String(50),
    "stadium": String(100),
    }
)

132

In [63]:
# Load weekly player stats
weekly = nfl.load_player_stats(seasons=True)
weekly_df = weekly.to_pandas()

In [64]:
# Load weekly player stats
weekly = nfl.load_player_stats(seasons=True)
weekly_df = weekly.to_pandas()

# Filter dataset for 2023 onward and drop rows with missing position
weekly_df = weekly_df[(weekly_df['season'] >= 2023) & (weekly_df['position_group'].isin(['QB', 'RB', 'WR', 'TE', 'SPEC'])) & (weekly_df['position'] != 'P')].dropna(subset=['position']).reset_index(drop=True)
weekly_df = weekly_df[['season', 'week', 'team', 'opponent_team', 'player_display_name', 'position']]

# Fix LA to LAR for consistency
weekly_df[['team', 'opponent_team']] = (
    weekly_df[['team', 'opponent_team']]
        .replace({'LA': 'LAR'})
)
# Rename player_display_name to player_name
weekly_df = weekly_df.rename(columns={"player_display_name": "player_name"})

# Strip periods from names
weekly_df['player_name'] = weekly_df['player_name'].str.replace('.', '', regex=False)

# Remove titles like Jr., Sr., III, etc.
weekly_df['player_name'] = weekly_df['player_name'].str.replace(r',?\s+(Jr|Sr|II|III|IV|V)$', '', regex=True)

# Fix specific name issues
name_corrections = {
    'Josh Palmer': 'Joshua Palmer',
    'Marquise Brown': 'Hollywood Brown',
    'Audric Estimé': 'Audric Estime'
}

weekly_df['player_name'] = weekly_df['player_name'].replace(name_corrections)

In [66]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE player_info.roster_reference RESTART IDENTITY CASCADE;"))

weekly_df.to_sql(
    "roster_reference",
    engine,
    "player_info",
    if_exists="append",
    index=False,
    dtype={
    "season": Integer(),
    "week": Integer(),
    "team": String(10),
    "opponent_team": String(10),
    "player_name": String(50),
    "position": String(10)
    }
)

175

## Historical Data

### Leagues

In [31]:
# Available Leagues
for year in (2023, 2024, 2025, 2026):
    league_ids = gm.league_ids(year)

    for lid in league_ids:
        lg = gm.to_league(lid)
        settings = lg.settings()  # includes league name + details
        print(f"League ID: {lid} | Name: {settings['name']} | Season: {settings['season']}")

RuntimeError: b'{\n    "error": {\n        "xml:lang": "en-us",\n        "yahoo:uri": "\\/fantasy\\/v2\\/users;use_login=1\\/games\\/teams?format=json",\n        "description": "This application is not authorized to perform this action.",\n        "detail": ""\n    }\n}'

In [19]:
league_ids = gm.league_ids()

for lid in league_ids:
    lg = gm.to_league(lid)



RuntimeError: b'{\n    "error": {\n        "xml:lang": "en-us",\n        "yahoo:uri": "\\/fantasy\\/v2\\/users\\/games\\/leagues?use_login=1&amp;is_available=0&amp;game_types=&amp;game_codes=&amp;seasons=&amp;format=json",\n        "description": "This application is not authorized to perform this action.",\n        "detail": ""\n    }\n}'

### Matchups, Managers, and Rosters

In [ ]:
# Populate this array with each team's data for each matchup
rows = []
managers = []

# Iterate over each league and each week to get all matchups
for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    
    for week in range(1, end_week + 1):
        matchup = lg.matchups(week)["fantasy_content"]["league"][1]['scoreboard']
        for key, value in matchup['0']['matchups'].items():
            if key != 'count':
                match = value['matchup']
                if match['is_matchup_recap_available'] != 0:
                    week = match['week']
                    week_start = match['week_start']
                    week_end = match['week_end']
                    isPlayoff = match['is_playoffs']
                    teams = match['0']['teams']
                    
                    team1 = teams['0']['team'][0][2]['name']
                    team2 = teams['1']['team'][0][2]['name']

                    points1 = float(teams['0']['team'][1]['team_points']['total'])
                    points2 = float(teams['1']['team'][1]['team_points']['total'])

                    # Chat
                    recap_url = match['matchup_recap_url']
                    week = int(match['week'])
                    winner_key = match['winner_team_key']

                    # Each matchup has exactly two teams: '0' and '1'
                    for side in ['0', '1']:
                        team = teams[side]['team']
                        team_data = team[0]   # long nested metadata list
                        scoring = team[1]     # dict with points and projections

                        # --- Extract basic info ---
                        team_key = team_data[0]['team_key']
                        team_id = team_data[1]['team_id']
                        team_name = team_data[2]['name']
                        points = float(scoring['team_points']['total'])
                        proj_points = float(scoring['team_projected_points']['total'])

                        # --- Winner flag ---
                        winner = team_key == winner_key

                        # --- Extract manager info ---
                        manager_info = None
                        for item in team_data:
                            if isinstance(item, dict) and 'managers' in item:
                                manager_info = item['managers'][0]['manager']
                                break

                        manager_id = manager_info.get('manager_id') if manager_info else None
                        nickname = manager_info.get('nickname') if manager_info else None
                        guid = manager_info.get('guid') if manager_info else None

                        if week == 1:
                            num_moves, num_trades = None, None
                            for item in team_data:
                                if isinstance(item, dict):
                                    if 'number_of_moves' in item:
                                        num_moves = int(item['number_of_moves'])
                                    elif 'number_of_trades' in item:
                                        try:
                                            num_trades = int(item['number_of_trades'])
                                        except ValueError:
                                            num_trades = None
                            managers.append({
                                "manager_guid": guid,
                                "season": year,
                                "league_id": league_id,
                                "team_id": team_id,
                                "team_key": team_key,
                                "number_of_moves": num_moves,
                                "number_of_trades": num_trades,
                                'manager_nickname': nickname
                            })

                        # --- Add a flattened record ---
                        rows.append({
                            "season": year,
                            "week": week,
                            "week_start": week_start,
                            "week_end": week_end,
                            "is_playoff": isPlayoff,
                            "matchup_id": key,
                            "team_id": team_id,
                            "team_name": team_name,
                            "team_key": team_key,
                            "manager_id": manager_id,
                            "manager_nickname": nickname,
                            "manager_guid": guid,
                            "points": points,
                            "projected_points": proj_points,
                            "winner": winner,
                            "recap_url": recap_url
                        })

matchups_df = pd.DataFrame(rows)
manager_seasons_df = pd.DataFrame(managers)
managers_df = manager_seasons_df[['manager_guid', 'manager_nickname']].drop_duplicates()
manager_seasons_df = manager_seasons_df.drop(columns=['manager_nickname'])

In [45]:
matchups_df.tail(5)

,season,week,week_start,week_end,is_playoff,matchup_id,team_id,team_name,team_key,manager_id,manager_nickname,manager_guid,points,projected_points,winner,recap_url
449,2025,13,2025-11-25,2025-12-01,0,2,8,Grit,461.l.1390291.t.8,8,Matthew,GOBJY4A3RDGQYERM2ETTMRC7YY,106.92,139.03,False,https://football.fantasysports.yahoo.com/f1/13...
450,2025,13,2025-11-25,2025-12-01,0,3,5,Amon Ra Njigba,461.l.1390291.t.5,5,Billy,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,120.92,153.99,False,https://football.fantasysports.yahoo.com/f1/13...
451,2025,13,2025-11-25,2025-12-01,0,3,9,Big 4🔒,461.l.1390291.t.9,9,Liam,BYOKM33VSC7PSCEB74CKX2E6FQ,124.84,144.08,True,https://football.fantasysports.yahoo.com/f1/13...
452,2025,13,2025-11-25,2025-12-01,0,4,6,Hail Mary john,461.l.1390291.t.6,6,john,BB5X53X74V3GT4BZO4W4ZE3S2I,110.64,132.91,False,https://football.fantasysports.yahoo.com/f1/13...
453,2025,13,2025-11-25,2025-12-01,0,4,7,Wire Warriors,461.l.1390291.t.7,7,Joseph,ZFECT2SSXQHCUOGITGLWEYRHJE,130.00,137.27,True,https://football.fantasysports.yahoo.com/f1/13...


In [46]:
manager_seasons_df.head(5)

,manager_guid,season,league_id,team_id,team_key,number_of_moves,number_of_trades
0,RLDAQUDDIH63SCRVGDUH7XHVVE,2023,423.l.927065,1,423.l.927065.t.1,65,10
1,GOBJY4A3RDGQYERM2ETTMRC7YY,2023,423.l.927065,5,423.l.927065.t.5,15,0
2,HL2OCHLAPFDTIFOI3ZENZETP5I,2023,423.l.927065,2,423.l.927065.t.2,33,3
3,2LZKPL56AY3GJ5QCXA6HWZOMGA,2023,423.l.927065,8,423.l.927065.t.8,21,5
4,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,2023,423.l.927065,3,423.l.927065.t.3,12,5


In [ ]:
all_players = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for week in range(1, end_week + 1):
        for team_name, team in teams.items():
            roster = lg.to_team(team_name).roster(week=week)
            playerIds = [player["player_id"] for player in roster]
            stats = lg.player_stats(playerIds, req_type='week', week=week)            
            for player, stat in zip(roster, stats):
                all_players.append({
                    "season": year,
                    "week": week,
                    "team_key": team_name,
                    "team_name": team['name'],
                    "manager_guid": team['managers'][0]['manager']['guid'],
                    "manager_name": team['managers'][0]['manager']['nickname'],
                    "player_id": player["player_id"],
                    "player_name": player["name"],
                    "position": player["selected_position"],
                    "points": stat['total_points'],
                })

roster = pd.DataFrame(all_players)

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE managers RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE manager_seasons RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE matchups RESTART IDENTITY CASCADE;"))

managers_df.to_sql('managers', engine, if_exists='append', index=False, dtype={
    "manager_guid": String(50),      # unique Yahoo GUID (e.g., 'GOBJY4A3RDGQYERM2ETTMRC7YY')
    "manager_nickname": String(50),  # manager's nickname ('Matthew')
    })

manager_seasons_df.to_sql('manager_seasons', engine, if_exists='replace', index=False, dtype={
    "manager_guid": String(50),      # unique Yahoo GUID (e.g., 'GOBJY4A3RDGQYERM2ETTMRC7YY')
    "season": Integer(),             # Season year
    "league_id": String(50),         # numeric league ID
    "team_id": Integer(),            # numeric team ID
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "number_of_moves": Integer(),    # total moves made
    "number_of_trades": Integer()    # total trades made
    })

matchups_df.to_sql('matchups', engine, if_exists='append', index=False, dtype={
    "season": Integer(),            # Season year
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "points": Float(),              # actual fantasy points scored
    "projected_points": Float(),    # projected points for the matchup
    "winner": String(10),           # 'win', 'loss', 'tie', or None
    "recap_url": String(255),       # URL to Yahoo recap (can be long)
})

roster.to_sql('roster', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),          # Season year
    "week": Integer(),            # week number (1–18)
    "team_key": String(50),       # Yahoo team key (short ID)
    "team_name": String(100),     # team name ("Hail Mary john")
    "manager_guid": String(50),   # unique Yahoo GUID ("GOBJY4A3RDGQYERM2ETTMRC7YY")
    "manager_name": String(50),   # nickname of manager ("Matthew")
    "player_id": Integer(),       # numeric Yahoo player ID (e.g., 31002)
    "player_name": String(100),   # player full name ("Lamar Jackson")
    "position": String(10),       # e.g., QB, RB, WR, TE, etc.
    "points": Float(),            # fantasy points (decimal)
})

454

### Transactions

In [ ]:
parsed_trades = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('trade', '1000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0]

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']

            from_team = transaction_info['source_team_name']
            from_team_key = transaction_info['source_team_key']
            to_team = transaction_info['destination_team_name']
            to_team_key = transaction_info['destination_team_key']

            parsed_trades.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'from_team': from_team,
                'from_team_key': from_team_key,
                'to_team': to_team,
                'to_team_key': to_team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

trades = pd.DataFrame(parsed_trades)

# Convert timestamp to datetime
trades['timestamp'] = pd.to_datetime(trades['timestamp'], unit='s')

In [ ]:
trades.to_sql('trades', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),            # year like 2025
    "league_id": String(50),        # league key (short ID)
    "trade_id": Integer(),          # trade or transaction ID
    "timestamp": DateTime,          # UNIX timestamp of transaction
    "status": String(50),           # e.g., 'successful'
    "from_team": String(100),       # name of team initiating transaction
    "from_team_key": String(100),   # key of team initiating transaction
    "to_team": String(100),         # name of team receiving player
    "to_team_key": String(100),     # key of team receiving player
    "player": String(100),          # player name
    "player_id": Integer(),         # player ID
    "pos": String(10),              # player position, e.g., 'DEF'
    "team_abbr": String(10)         # team abbreviation, e.g., 'CHI'
})

237

In [13]:
parsed_waivers = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('add', '5000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0] if isinstance(player_entry['player'][1]['transaction_data'], list) else player_entry['player'][1]['transaction_data']

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']
            type = transaction_info['type']
            add = type == 'add'
            teamType = 'destination' if add else 'source'

            team = transaction_info[f'{teamType}_team_name'] 
            team_key = transaction_info[f'{teamType}_team_key']

            parsed_waivers.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'type': type,
                'team': team,
                'team_key': team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

waivers = pd.DataFrame(parsed_waivers)

In [ ]:
# Convert timestamp to datetime and player_id to int
waivers['timestamp'] = pd.to_datetime(waivers['timestamp'], unit='s')
waivers['player_id'] = waivers['player_id'].astype(int)

In [ ]:
waivers.to_sql('waivers', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),           # year like 2025
    "league_id": String(50),       # league key (short ID)
    "trade_id": String(50),        # trade or transaction ID
    "timestamp": DateTime,         # UNIX timestamp of transaction
    "status": String(50),          # e.g., 'successful'
    "type": String(50),            # transaction type, e.g., 'add/drop'
    "team": String(100),           # team name
    "team_key": String(100),       # team key
    "player": String(100),         # player name
    "player_id": Integer(),        # player ID
    "pos": String(10),             # player position, e.g., 'DEF'
    "team_abbr": String(10)        # team abbreviation, e.g., 'CHI'
})

737

In [98]:
rows = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for team_name, team in teams.items(): 
        rows.append({
            'Season': year,
            'league_id': league_id,
            'team_key': team_name,
            'manager_id': team['managers'][0]['manager']['manager_id'],
            'nickname': team['managers'][0]['manager']['nickname'],
            'guid': team['managers'][0]['manager']['guid']
        })

teamsDf = pd.DataFrame(rows)

In [156]:
teamsDf.to_sql('teams', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),         # year like 2025
    "league_id": String(50),     # Yahoo team key (short ID)
    "team_key": String(100),     # team key 
    "manager_id": Integer(),     # managerid
    "nickname": String(15),      # nickname
    "guid": String(50)           # unique Yahoo GUID ("GOBJY4A3RDGQYERM2ETTMRC7YY")
})

30

### Detailed Roster

In [ ]:
all_players = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for week in range(1, end_week + 1):
        for team_name, team in teams.items():
            roster = lg.to_team(team_name).roster(week=week)
            playerIds = [player["player_id"] for player in roster]
            stats = lg.player_stats(playerIds, req_type='week', week=week)
            for player, stat in zip(roster, stats):
                all_players.append({
                    "season": year,
                    "week": week,
                    "team_key": team_name,
                    "team_name": team['name'],
                    "manager_guid": team['managers'][0]['manager']['guid'],
                    "manager_name": team['managers'][0]['manager']['nickname'],
                    "player_id": player["player_id"],
                    "player_name": player["name"],
                    "played_position": player['selected_position'],
                    "position_type": stat['position_type'],
                    "pass_yds": stat['Pass Yds'] if 'Pass Yds' in stat else 0,
                    "pass_td": stat['Pass TD'] if 'Pass TD' in stat else 0,
                    "pass_int": stat['Int'] if 'Int' in stat and player['position_type'] == 'O' else 0,
                    "rush_att": stat['Rush Att'] if 'Rush Att' in stat else 0,
                    "rush_yds": stat['Rush Yds'] if 'Rush Yds' in stat else 0,
                    "rush_td": stat['Rush TD'] if 'Rush TD' in stat else 0,
                    "targets": stat['Targets'] if 'Targets' in stat else 0,
                    "rec": stat['Rec'] if 'Rec' in stat else 0,
                    "rec_yds": stat['Rec Yds'] if 'Rec Yds' in stat else 0,
                    "rec_td": stat['Rec TD'] if 'Rec TD' in stat else 0,
                    "offensive_ret_td": stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'O' else 0,
                    "two_pt_conversion": stat['2-PT'] if '2-PT' in stat else 0,
                    "fum_lost": stat['Fum Lost'] if 'Fum Lost' in stat else 0,
                    "fum_ret_td": stat['Fum Ret TD'] if 'Fum Ret TD' in stat else 0,
                    'fg_0_19': stat['FG 0-19'] if 'FG 0-19' in stat else 0,
                    'fg_20_29': stat['FG 20-29'] if 'FG 20-29' in stat else 0,
                    'fg_30_39': stat['FG 30-39'] if 'FG 30-39' in stat else 0,
                    'fg_40_49': stat['FG 40-49'] if 'FG 40-49' in stat else 0,
                    'fg_50_plus': stat['FG 50+'] if 'FG 50+' in stat else 0,
                    'pat_made': stat['PAT Made'] if 'PAT Made' in stat else 0,
                    'pts_allow': stat['Pts Allow'] if 'Pts Allow' in stat else 0,
                    'sack': stat['Sack'] if 'Sack' in stat else 0,
                    'dt_int': stat['Int'] if 'Int' in stat and player['position_type'] == 'DT' else 0,
                    'fum_rec': stat['Fum Rec'] if 'Fum Rec' in stat else 0,
                    'dt_td': stat['TD'] if 'TD' in stat and player['position_type'] == 'DT' else 0,
                    'safe': stat['Safe'] if 'Safe' in stat else 0,
                    'blk_kick': stat['Blk Kick'] if 'Blk Kick' in stat else 0,
                    'dt_ret_td': stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'DT' else 0,
                    'pts_allow_0': stat['Pts Allow 0'] if 'Pts Allow 0' in stat else 0,
                    'pts_allow_1_6': stat['Pts Allow 1-6'] if 'Pts Allow 1-6' in stat else 0,
                    'pts_allow_7_13': stat['Pts Allow 7-13'] if 'Pts Allow 7-13' in stat else 0,
                    'pts_allow_14_20': stat['Pts Allow 14-20'] if 'Pts Allow 14-20' in stat else 0,
                    'pts_allow_21_27': stat['Pts Allow 21-27'] if 'Pts Allow 21-27' in stat else 0,
                    'pts_allow_28_34': stat['Pts Allow 28-34'] if 'Pts Allow 28-34' in stat else 0,
                    'pts_allow_35_plus': stat['Pts Allow 35+'] if 'Pts Allow 35+' in stat else 0,
                    'xpr': stat['XPR'] if 'XPR' in stat else 0,
                    "points": stat['total_points'] if 'total_points' in stat else 0,
                })

detailed_roster = pd.DataFrame(all_players)

In [5]:
# Strip periods from names
detailed_roster['player_name'] = detailed_roster['player_name'].str.replace('.', '', regex=False)

# Remove titles like Jr., Sr., III, etc.
detailed_roster['player_name'] = detailed_roster['player_name'].str.replace(r',?\s+(Jr|Sr|II|III|IV|V)$', '', regex=True)

In [13]:
detailed_roster.to_sql('detailed_roster', engine, if_exists='replace', index=False,
    dtype={
        "season": Integer(),
        "week": Integer(),
        "team_key": String(100),
        "team_name": String(100),
        "manager_guid": String(50),
        "manager_name": String(50),
        "player_id": Integer(),
        "player_name": String(100),
        "played_position": String(10),
        "position_type": String(5),

        # Passing
        "pass_yds": Integer(),
        "pass_td": Integer(),
        "pass_int": Integer(),

        # Rushing
        "rush_att": Integer(),
        "rush_yds": Integer(),
        "rush_td": Integer(),

        # Receiving
        "targets": Integer(),
        "rec": Integer(),
        "rec_yds": Integer(),
        "rec_td": Integer(),

        # Misc offensive
        "offensive_ret_td": Integer(),
        "two_pt_conversion": Integer(),
        "fum_lost": Integer(),
        "fum_ret_td": Integer(),

        # Kicker
        "fg_0_19": Integer(),
        "fg_20_29": Integer(),
        "fg_30_39": Integer(),
        "fg_40_49": Integer(),
        "fg_50_plus": Integer(),
        "pat_made": Integer(),
        "xpr": Integer(),

        # Defense
        "pts_allow": Integer(),
        "sack": Integer(),
        "dt_int": Integer(),
        "fum_rec": Integer(),
        "dt_td": Integer(),
        "safe": Integer(),
        "blk_kick": Integer(),
        "dt_ret_td": Integer(),

        # Defense point tiers
        "pts_allow_0": Integer(),
        "pts_allow_1_6": Integer(),
        "pts_allow_7_13": Integer(),
        "pts_allow_14_20": Integer(),
        "pts_allow_21_27": Integer(),
        "pts_allow_28_34": Integer(),
        "pts_allow_35_plus": Integer(),

        # Final fantasy points
        "points": Float()
    }
)

144

## Important Weekly Runs

In [27]:
currentSeason = max(list(idMap.keys()))

In [28]:
lg = gm.to_league(idMap[currentSeason])
settings = lg.settings()
current_week = lg.current_week()

### Current Matchup

In [6]:
rows = []
matchup = lg.matchups(current_week)["fantasy_content"]["league"][1]['scoreboard']

for key, value in matchup['0']['matchups'].items():
    if key != 'count':
        match = value['matchup']
        week = match['week']
        week_start = match['week_start']
        week_end = match['week_end']
        isPlayoff = match['is_playoffs']
        teams = match['0']['teams']
        
        team1 = teams['0']['team'][0][2]['name']
        team2 = teams['1']['team'][0][2]['name']

        # Each matchup has exactly two teams: '0' and '1'
        for side in ['0', '1']:
            team = teams[side]['team']
            team_data = team[0]   # long nested metadata list
            scoring = team[1]     # dict with points and projections

            # --- Extract basic info ---
            team_key = team_data[0]['team_key']
            team_id = team_data[1]['team_id']
            team_name = team_data[2]['name']
            proj_points = float(scoring['team_projected_points']['total'])

            manager_info = None
            for item in team_data:
                if isinstance(item, dict) and 'managers' in item:
                    manager_info = item['managers'][0]['manager']
                    break

            manager_id = manager_info.get('manager_id') if manager_info else None
            nickname = manager_info.get('nickname') if manager_info else None
            guid = manager_info.get('guid') if manager_info else None

            # --- Add a flattened record ---
            rows.append({
                "season": currentSeason,
                "week": week,
                "week_start": week_start,
                "week_end": week_end,
                "is_playoff": isPlayoff,
                "matchup_id": key,
                "team_id": team_id,
                "team_name": team_name,
                "team_key": team_key,
                "manager_id": manager_id,
                "manager_nickname": nickname,
                "manager_guid": guid,
                "projected_points": proj_points,
            })

currentMatchup = pd.DataFrame(rows)

In [7]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE manager_info.currentmatchup RESTART IDENTITY CASCADE;"))

In [8]:
currentMatchup.to_sql('currentmatchup', engine, 'manager_info', if_exists='append', index=False, dtype={
    "season": Integer(),            # Season
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "projected_points": Float()    # projected points for the matchup
})

8

### Append Last Weeks Data

In [29]:
previousWeek = current_week - 1 if current_week != 18 else 18

In [30]:
rows = []
matchup = lg.matchups(previousWeek)["fantasy_content"]["league"][1]['scoreboard']

for key, value in matchup['0']['matchups'].items():
    if key != 'count':
        match = value['matchup']
        if match['is_matchup_recap_available'] != 0:
            week = match['week']
            week_start = match['week_start']
            week_end = match['week_end']
            isPlayoff = match['is_playoffs']
            teams = match['0']['teams']
            
            team1 = teams['0']['team'][0][2]['name']
            team2 = teams['1']['team'][0][2]['name']

            points1 = float(teams['0']['team'][1]['team_points']['total'])
            points2 = float(teams['1']['team'][1]['team_points']['total'])

            # Chat
            recap_url = match['matchup_recap_url']
            week = int(match['week'])
            winner_key = match['winner_team_key']

            # Each matchup has exactly two teams: '0' and '1'
            for side in ['0', '1']:
                team = teams[side]['team']
                team_data = team[0]   # long nested metadata list
                scoring = team[1]     # dict with points and projections

                # --- Extract basic info ---
                team_key = team_data[0]['team_key']
                team_id = team_data[1]['team_id']
                team_name = team_data[2]['name']
                points = float(scoring['team_points']['total'])
                proj_points = float(scoring['team_projected_points']['total'])

                # --- Winner flag ---
                winner = team_key == winner_key

                # --- Extract manager info ---
                manager_info = None
                for item in team_data:
                    if isinstance(item, dict) and 'managers' in item:
                        manager_info = item['managers'][0]['manager']
                        break

                manager_id = manager_info.get('manager_id') if manager_info else None
                nickname = manager_info.get('nickname') if manager_info else None
                guid = manager_info.get('guid') if manager_info else None

                if previousWeek == 1:
                    num_moves, num_trades = None, None
                    for item in team_data:
                        if isinstance(item, dict):
                            if 'number_of_moves' in item:
                                num_moves = int(item['number_of_moves'])
                            elif 'number_of_trades' in item:
                                try:
                                    num_trades = int(item['number_of_trades'])
                                except ValueError:
                                    num_trades = None
                    managers.append({
                        "manager_guid": guid,
                        "season": currentSeason,
                        "league_id": league_id,
                        "manager_id": manager_id,
                        "manager_nickname": nickname,
                        "team_id": team_id,
                        "number_of_moves": num_moves,
                        "number_of_trades": num_trades
                    })

                # --- Add a flattened record ---
                rows.append({
                    "season": currentSeason,
                    "week": week,
                    "week_start": week_start,
                    "week_end": week_end,
                    "is_playoff": isPlayoff,
                    "matchup_id": key,
                    "team_id": team_id,
                    "team_name": team_name,
                    "team_key": team_key,
                    "manager_id": manager_id,
                    "manager_nickname": nickname,
                    "manager_guid": guid,
                    "points": points,
                    "projected_points": proj_points,
                    "winner": winner,
                    "recap_url": recap_url
                })

lastWeekMatchup = pd.DataFrame(rows)
lastWeekMatchup['game_quality'] = pd.cut(lastWeekMatchup['points'], bins =[0, 120, 150, float('inf')], labels = ['poor', 'average', 'strong'])

In [31]:
if previousWeek == 1:
    manager_seasons_df = pd.DataFrame(managers)
    managers_df = manager_seasons_df[['manager_guid', 'manager_nickname']].drop_duplicates()
    manager_seasons_df = manager_seasons_df.drop(columns=['manager_nickname'])

    # Existing managers
    manager_current = pd.read_sql("SELECT * FROM manager_info.managers", engine)

    # Get any new managers
    managers_new = managers_df[
        ~managers_df['manager_guid'].isin(manager_current['manager_guid'])
    ]
    start = time.time()
    managers_new.to_sql("managers", engine, 'manager_info', if_exists="append", index=False)
    manager_seasons_df.to_sql("manager_seasons", engine, 'manager_info', if_exists="append", index=False)
    end = time.time()
    print(f"New managers and manager seasons insertion completed in {end - start:.2f} seconds")

In [32]:
if previousWeek != 0:
    start = time.time()
    
    lastWeekMatchup.to_sql('matchups', engine, 'manager_info', if_exists='append', index=False, dtype={
        "season": Integer(),            # Season year
        "week": Integer(),              # week number (1–18)
        "week_start": Date(),           # start date of the week
        "week_end": Date(),             # end date of the week
        "is_playoff": Integer(),        # True if this week is in playoffs
        "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
        "team_id": Integer(),           # numeric team ID
        "team_name": String(100),       # e.g., "Hail Mary john"
        "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
        "manager_id": Integer(),        # numeric manager ID (if available)
        "manager_nickname": String(50), # e.g., "Matthew"
        "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
        "points": Float(),              # actual fantasy points scored
        "projected_points": Float(),    # projected points for the matchup
        "winner": String(10),           # 'win', 'loss', 'tie', or None
        "recap_url": String(255),       # URL to Yahoo recap (can be long)
        "game_quality": String(10)      # Designation of strong, average, or poor game
    })

    end = time.time()
    print(f"Insertion completed in {end - start:.2f} seconds")

Insertion completed in 0.10 seconds


In [33]:
if previousWeek != 0:
    previousWeekPlayers = []
    teams = lg.teams()

    for team_name, team in teams.items():
        roster = lg.to_team(team_name).roster(week=previousWeek)
        playerIds = [player["player_id"] for player in roster]
        stats = lg.player_stats(playerIds, req_type='week', week=previousWeek)            
        for player, stat in zip(roster, stats):
            previousWeekPlayers.append({
                        "season": currentSeason,
                        "week": previousWeek,
                        "team_key": team_name,
                        "team_name": team['name'],
                        "manager_guid": team['managers'][0]['manager']['guid'],
                        "manager_name": team['managers'][0]['manager']['nickname'],
                        "player_id": player["player_id"],
                        "player_name": player["name"],
                        "played_position": player['selected_position'],
                        "position_type": stat['position_type'],
                        "pass_yds": stat['Pass Yds'] if 'Pass Yds' in stat else 0,
                        "pass_td": stat['Pass TD'] if 'Pass TD' in stat else 0,
                        "pass_int": stat['Int'] if 'Int' in stat and player['position_type'] == 'O' else 0,
                        "rush_att": stat['Rush Att'] if 'Rush Att' in stat else 0,
                        "rush_yds": stat['Rush Yds'] if 'Rush Yds' in stat else 0,
                        "rush_td": stat['Rush TD'] if 'Rush TD' in stat else 0,
                        "targets": stat['Targets'] if 'Targets' in stat else 0,
                        "rec": stat['Rec'] if 'Rec' in stat else 0,
                        "rec_yds": stat['Rec Yds'] if 'Rec Yds' in stat else 0,
                        "rec_td": stat['Rec TD'] if 'Rec TD' in stat else 0,
                        "offensive_ret_td": stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'O' else 0,
                        "two_pt_conversion": stat['2-PT'] if '2-PT' in stat else 0,
                        "fum_lost": stat['Fum Lost'] if 'Fum Lost' in stat else 0,
                        "fum_ret_td": stat['Fum Ret TD'] if 'Fum Ret TD' in stat else 0,
                        'fg_0_19': stat['FG 0-19'] if 'FG 0-19' in stat else 0,
                        'fg_20_29': stat['FG 20-29'] if 'FG 20-29' in stat else 0,
                        'fg_30_39': stat['FG 30-39'] if 'FG 30-39' in stat else 0,
                        'fg_40_49': stat['FG 40-49'] if 'FG 40-49' in stat else 0,
                        'fg_50_plus': stat['FG 50+'] if 'FG 50+' in stat else 0,
                        'pat_made': stat['PAT Made'] if 'PAT Made' in stat else 0,
                        'pts_allow': stat['Pts Allow'] if 'Pts Allow' in stat else 0,
                        'sack': stat['Sack'] if 'Sack' in stat else 0,
                        'dt_int': stat['Int'] if 'Int' in stat and player['position_type'] == 'DT' else 0,
                        'fum_rec': stat['Fum Rec'] if 'Fum Rec' in stat else 0,
                        'dt_td': stat['TD'] if 'TD' in stat and player['position_type'] == 'DT' else 0,
                        'safe': stat['Safe'] if 'Safe' in stat else 0,
                        'blk_kick': stat['Blk Kick'] if 'Blk Kick' in stat else 0,
                        'dt_ret_td': stat['Ret TD'] if 'Ret TD' in stat and player['position_type'] == 'DT' else 0,
                        'pts_allow_0': stat['Pts Allow 0'] if 'Pts Allow 0' in stat else 0,
                        'pts_allow_1_6': stat['Pts Allow 1-6'] if 'Pts Allow 1-6' in stat else 0,
                        'pts_allow_7_13': stat['Pts Allow 7-13'] if 'Pts Allow 7-13' in stat else 0,
                        'pts_allow_14_20': stat['Pts Allow 14-20'] if 'Pts Allow 14-20' in stat else 0,
                        'pts_allow_21_27': stat['Pts Allow 21-27'] if 'Pts Allow 21-27' in stat else 0,
                        'pts_allow_28_34': stat['Pts Allow 28-34'] if 'Pts Allow 28-34' in stat else 0,
                        'pts_allow_35_plus': stat['Pts Allow 35+'] if 'Pts Allow 35+' in stat else 0,
                        'xpr': stat['XPR'] if 'XPR' in stat else 0,
                        "points": stat['total_points'] if 'total_points' in stat else 0,
                    })

    previousWeekRoster = pd.DataFrame(previousWeekPlayers)

In [34]:
start = time.time()
previousWeekRoster.to_sql('detailed_roster', engine, 'player_info', if_exists='append', index=False, chunksize=200, dtype={
        "season": Integer(),
        "week": Integer(),
        "team_key": String(100),
        "team_name": String(100),
        "manager_guid": String(50),
        "manager_name": String(50),
        "player_id": Integer(),
        "player_name": String(100),
        "played_position": String(10),
        "position_type": String(5),

        # Passing
        "pass_yds": Integer(),
        "pass_td": Integer(),
        "pass_int": Integer(),

        # Rushing
        "rush_att": Integer(),
        "rush_yds": Integer(),
        "rush_td": Integer(),

        # Receiving
        "targets": Integer(),
        "rec": Integer(),
        "rec_yds": Integer(),
        "rec_td": Integer(),

        # Misc offensive
        "offensive_ret_td": Integer(),
        "two_pt_conversion": Integer(),
        "fum_lost": Integer(),
        "fum_ret_td": Integer(),

        # Kicker
        "fg_0_19": Integer(),
        "fg_20_29": Integer(),
        "fg_30_39": Integer(),
        "fg_40_49": Integer(),
        "fg_50_plus": Integer(),
        "pat_made": Integer(),
        "xpr": Integer(),

        # Defense
        "pts_allow": Integer(),
        "sack": Integer(),
        "dt_int": Integer(),
        "fum_rec": Integer(),
        "dt_td": Integer(),
        "safe": Integer(),
        "blk_kick": Integer(),
        "dt_ret_td": Integer(),

        # Defense point tiers
        "pts_allow_0": Integer(),
        "pts_allow_1_6": Integer(),
        "pts_allow_7_13": Integer(),
        "pts_allow_14_20": Integer(),
        "pts_allow_21_27": Integer(),
        "pts_allow_28_34": Integer(),
        "pts_allow_35_plus": Integer(),

        # Final fantasy points
        "points": Float()
    }
)
end = time.time()
print(f"Insertion completed in {end - start:.2f} seconds")

IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "detailed_roster_pkey"
DETAIL:  Key (season, week, manager_guid, player_id)=(2025, 16, BHMUHFWNWP25IWXVETH4HHI7AY, 40881) already exists.

[SQL: INSERT INTO player_info.detailed_roster (season, week, team_key, team_name, manager_guid, manager_name, player_id, player_name, played_position, position_type, pass_yds, pass_td, pass_int, rush_att, rush_yds, rush_td, targets, rec, rec_yds, rec_td, o ... 151469 characters truncated ... w_21_27__164)s, %(pts_allow_28_34__164)s, %(pts_allow_35_plus__164)s, %(xpr__164)s, %(points__164)s)]
[parameters: {'rec_td__0': 0.0, 'fg_40_49__0': 0.0, 'pass_td__0': 2.0, 'fg_30_39__0': 0.0, 'pat_made__0': 0.0, 'sack__0': 0.0, 'fg_50_plus__0': 0.0, 'pts_allow_14_20__0': 0.0, 'team_name__0': 'Dust Off Frankie', 'rush_att__0': 10.0, 'pts_allow_35_plus__0': 0.0, 'manager_name__0': 'Frankie', 'dt_td__0': 0.0, 'player_name__0': 'Drake Maye', 'fum_ret_td__0': 0.0, 'pts_a

In [35]:
sheet_week = f'Week {previousWeek} - {currentSeason}'
parlays = get_parlays(sheet_week)
parlays['season'] = currentSeason
parlays['week'] = previousWeek
parlays.columns = parlays.columns.str.strip()  # Remove leading/trailing whitespace from column names
parlays = parlays[['season', 'week', 'Manager', 'Bet', 'Hit']]
parlays.rename(columns={'Manager': 'manager', 'Bet': 'bet', 'Hit': 'hit'}, inplace=True)

parlays.to_sql('parlays', engine, schema='transactions', if_exists='append', index=False, dtype={
    "season": Integer(),            # Season year
    "week": Integer(),              # week number (1–18)
    "manager": String(50),          # manager nickname
    "bet": String(100),             # bet description
    "hit": String(20),              # 'Yes' or 'No'
})

9

## Reports

In [3]:
sql = text('''
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
''')

report = pd.read_sql(sql, con=engine.connect())

In [4]:
for index, row in report.iterrows():
    if index == 0: print(f"Week {row['week']} of the {row['season']} season")
    print(f"""
    - {row['projectedwinningmanager']} vs {row['projectedlosingmanager']} 
        {f"- Lifetime record ({row['lifetimewins']}-{row['lifetimelosses']})" if not pd.isna(row['season_s']) else '-First matchup'}
        {f"- Smallest point differential {row['smallestpointdifferential']} ({row['winningmanager_s']} beat {row['losingmanager_s']} {row['winningpoints_s']} to {row['losingpoints_s']} in week {int(row['week_s'])} of the {row['season_s']} season)" if not pd.isna(row['season_s']) else ''}
        {f"- Largest point differential {row['largestpointdifferential']} ({row['winningmanager_l']} beat {row['losingmanager_l']} {row['winningpoints_l']} to {row['losingpoints_l']} in week {int(row['week_l'])} of the {row['season_l']} season)" if not pd.isna(row['season_s']) else ''}""")

Week 3 of the 2026 season

    - Billy vs Joseph 
        - Lifetime record (4-1)
        - Smallest point differential 4.46 (Joseph beat Billy 144.06 to 139.6 in week 9 of the 2024 season)
        - Largest point differential 62.9 (Billy beat Joseph 180.28 to 117.38 in week 13 of the 2023 season)

    - Chris vs John 
        - Lifetime record (3-3)
        - Smallest point differential 5.52 (John beat Chris 135.36 to 129.84 in week 14 of the 2023 season)
        - Largest point differential 79.64 (Chris beat John 184.8 to 105.16 in week 9 of the 2024 season)

    - Duffy vs Liam 
        - Lifetime record (1-2)
        - Smallest point differential 5.84 (Liam beat Duffy 145.76 to 139.92 in week 3 of the 2025 season)
        - Largest point differential 29.04 (Duffy beat Liam 154.42 to 125.38 in week 12 of the 2025 season)

    - Frankie vs Skyler 
        - Lifetime record (3-4)
        - Smallest point differential 0.46 (Skyler beat Frankie 141.04 to 140.58 in week 5 of the 2024 sea

- Would be cool to see most recent match up and current win streaks.

## SQL Work

In [ ]:
--SQL: View Matchup History
Select Season, Week, Playoffs, WinningManager, WinningPoints, LosingManager, LosingPoints, Min(PointDifferential) as PointDiff 
From (
    SELECT to_char(w.week_start, 'YYYY') as Season, w.week, w.is_playoff as Playoffs, w.team_name as WinningTeam, wm.propername as WinningManager, w.points as WinningPoints, l.team_name as LosingTeam, wl.propername as LosingManager, l.points as LosingPoints, 
    (ROUND((w.points - l.points)::numeric, 3)) as PointDifferential
    FROM public.matchups W Left Join public.manager_map wm on w.manager_nickname = wm.manager_nickname
    Left Join public.matchups L On W.week = L.Week And W.Week_start = L.Week_Start and W.Week_end = L.Week_end and W.matchup_id = L.matchup_id and L.Winner = 'false'
    Left Join public.manager_map wl on l.manager_nickname = wl.manager_nickname
    Where W.Winner = 'true'
)
group By Season, Week, Playoffs, WinningManager, WinningPoints, LosingManager, LosingPoints
Order By PointDiff Desc
;

In [ ]:
-- Biggest Fleeces
With TradeTotalPoints as(
SELECT T.Season, T.Trade_ID, T.to_team as Team, T.to_team_key as Team_Key, Round(Avg(P.points)::numeric, 4) as Points
FROM public.trades T
Left Join SeasonalPoints P On t.season = p.season and t.player_id = p.player_id
Where T.status = 'successful'
Group By T.Season, T.Trade_ID, T.to_team, T.to_team_key
) 
Select F.Season, F.Trade_ID, F.Team as Team_One, F.Team_Key as Team_One_Key, F.Points as Team_One_Points, S.Team as Team_Two, S.Team_Key as Team_Two_Key, S.Points as Team_Two_Key,
F.Points - S.Points as Differential
From TradeTotalPoints F
Left Join TradeTotalPoints S On F.Season = S.Season and F.Trade_ID = S.Trade_ID and F.Team_Key != S.Team_Key
Where F.Points - S.Points > 0
Group By F.Season, F.Trade_ID, F.Team, F.Team_Key, F.Points, S.Team, S.Team_Key, S.Points
Order By Differential Desc
;

In [ ]:
-- Weekly Report
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L
From (
    Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
    from vw_currentmatchup M
    Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
    Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
    Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
    Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
;

In [ ]:
-- Not done, adding Most recent match up
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L,
HLAST.Season as Season_Last,
HLAST.Week as Week_Last,
HLAST.winningmanager as WinningManager_Last,
HLAST.winningpoints as WinningPoints_Last,
HLAST.losingmanager as LosingManager_Last,
HLAST.losingpoints as LosingPoints_Last
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
Left Join matchuphistory HLAST On 
    (S.projectedwinningmanager = HLAST.winningmanager or S.projectedwinningmanager = HLAST.losingmanager)
    and (S.projectedlosingmanager = HLAST.losingmanager or S.projectedlosingmanager = HLAST.winningmanager)
    and HLAST.Week = (
        Select Max(H2.Week)
        From matchuphistory H2
        Where 
            (S.projectedwinningmanager = H2.winningmanager or S.projectedwinningmanager = H2.losingmanager)
            and (S.projectedlosingmanager = H2.losingmanager or S.projectedlosingmanager = H2.winningmanager)
            and H2.Season = (
                Select Max(H3.Season)
                From matchuphistory H3
                Where 
                    (S.projectedwinningmanager = H3.winningmanager or S.projectedwinningmanager = H3.losingmanager)
                    and (S.projectedlosingmanager = H3.losingmanager or S.projectedlosingmanager = H3.winningmanager)
            )
    )

In [ ]:
-- Historical Standings by Manager
Select map.propername, Sum(Case When winner = 'true' Then 1 Else 0 End) as wins,
       Sum(Case When winner = 'false' Then 1 Else 0 End) as losses
FROM manager_info.matchups m
LEFT JOIN manager_map map ON m.manager_nickname::text = map.manager_nickname::text
WHERE (season = 2025 and week < 5)
and m.is_playoff = 0
GROUP BY map.propername
ORDER BY wins DESC

In [ ]:
Select * 
From manager_info.matchups
Where week = 17
and (season, manager_guid) in (
    Select season, manager_guid From manager_info.matchups
    Where week = 16
    and (season, matchup_id) in (
            Select season, matchup_id 
            From manager_info.matchups
            Where week = 16
            and (season, manager_guid) in (
                    Select season, manager_guid
                    From manager_info.matchups
                    Where week = 15
                    and winner = 'true'
            )
    )
and winner = 'true'
)
Order By Season